In [1]:
%load_ext autoreload
%autoreload 2
import sys, os
sys.path.insert(0, os.path.abspath("C:/Users/Voror/Projects/Personal/sep"))
from sklearn.model_selection import KFold
from SIDER_dataset.libraries.XofN_library import *
from SIDER_dataset.libraries.PCT_library import run_PCT
from SIDER_dataset.libraries.feature_evaluation_methods import feature_variance_reduction_scores
from SIDER_dataset.libraries.utils import get_clus_path


In [2]:
# Set ADR to predict and scoring
clus_path = get_clus_path()
paths = get_dataset_paths()
print(len(paths), "datasets")
paths

18 datasets


[{'dataset_path': 'C:\\Users\\Voror\\Projects\\Personal/sep/SIDER_dataset/datasets/clean_multi_label_datasets/CPI+fingerprint_cardiac.csv',
  'dataset_name': 'CPI+fingerprint_cardiac',
  'label_set': ['se_C0016382',
   'se_C0018799',
   'se_C0003811',
   'se_C0428977',
   'se_C0027051',
   'se_C0018790'],
  'features': 2147,
  'original_features': ['cpi_9606.ENSP00000000442',
   'cpi_9606.ENSP00000001008',
   'cpi_9606.ENSP00000003084',
   'cpi_9606.ENSP00000003100',
   'cpi_9606.ENSP00000005178',
   'cpi_9606.ENSP00000011292',
   'cpi_9606.ENSP00000011653',
   'cpi_9606.ENSP00000012443',
   'cpi_9606.ENSP00000013034',
   'cpi_9606.ENSP00000014930',
   'cpi_9606.ENSP00000019103',
   'cpi_9606.ENSP00000023897',
   'cpi_9606.ENSP00000039007',
   'cpi_9606.ENSP00000044462',
   'cpi_9606.ENSP00000054668',
   'cpi_9606.ENSP00000078429',
   'cpi_9606.ENSP00000155840',
   'cpi_9606.ENSP00000164139',
   'cpi_9606.ENSP00000171757',
   'cpi_9606.ENSP00000176183',
   'cpi_9606.ENSP00000176195',
 

In [3]:
paths = [path for path in paths if "ten_mid" in path["dataset_name"]]
len(paths)

3

In [4]:
k = 10
random_state = 42
performances = []
ranking_criteria = "MDI"  # or "VAR"
include_original_features_options = [True, False]
training_algorithm = "Jaccard"
eval_criteria = ["averageAUROC", "HammingLoss", "SubsetAccuracy", "RankingLoss", "MacroPrecision", "MacroRecall",
                 "MacroFOne"]
max_size = 5
cv_results = []

for idx, path in enumerate(paths, start=1):
    run_config = f"\n--- Running with label:'{path["label_set"]}' training_algorithm:'{training_algorithm}' eval_criterion:'{eval_criteria}' max_size:'{max_size}' ---"
    print(run_config)
    run_config_name = "_".join(
        [
            path["dataset_name"],
            training_algorithm,
            "_".join(eval_criteria),
            str(max_size),
        ]
    )
    logging_path = f"XofN_jaccard/logs/{run_config_name}_logs.txt"
    print(f"Logs can be found in {logging_path}.")
    logger = get_logger(logging_path)
    logger.info(run_config_name)

    # Load dataset
    current_df = pd.read_csv(path["dataset_path"])
    features = get_features(current_df, path["label_set"])
    # current_df = current_df[features[:10] + path["label_set"]]
    kf = KFold(n_splits=k, shuffle=True, random_state=random_state)
    for fold, (train_idx, test_idx) in enumerate(kf.split(current_df), start=1):
        print(f"\nFold {fold}/{k} ({path["dataset_name"]} {idx}/{len(paths)})")
        train_dataset = current_df.iloc[train_idx]
        test_dataset = current_df.iloc[test_idx]

        if ranking_criteria == "VAR":
            feature_rankings = feature_variance_reduction_scores(train_dataset[features],
                                                                 train_dataset[path["label_set"]])
        elif ranking_criteria == "MDI":
            feature_rankings = calculate_mdi_multi_rf(train_dataset, path["label_set"])
        else:
            raise NotImplementedError

        XofN_groupings, avg_features, gen_XofN_time = generate_XofN_list_multi_jaccard(
            train_dataset,
            feature_rankings,
            max_size,
            logger
        )
        if len(XofN_groupings) == 0:
            print("no XofN groupings were created")
        else:
            for include_original_features in include_original_features_options:
                current_train_dataset = group_features(
                    train_dataset,
                    path["label_set"],
                    XofN_groupings,
                    include_original_features,
                    verbose=True
                )

                current_test_dataset = group_features(
                    test_dataset,
                    path["label_set"],
                    XofN_groupings,
                    include_original_features,
                    verbose=True
                )

                current_train_dataset.to_csv(f"XofN_jaccard/tmp/train_dataset.csv", index=False)
                current_test_dataset.to_csv(f"XofN_jaccard/tmp/test_dataset.csv", index=False)

                training_start = time.perf_counter()
                original_res, pruned_res, training_time = run_PCT(clus_path,
                                                                  "XofN_jaccard/tmp/train_dataset.csv",
                                                                  path["label_set"],
                                                                  eval_criteria,
                                                                  test_dataset_path=f"XofN_jaccard/tmp/test_dataset.csv")
                pruned_performance = get_fold_results(pruned_res, eval_criteria, True, fold, include_original_features,
                                                      XofN_groupings,
                                                      gen_XofN_time,
                                                      training_time, path["dataset_name"])
                performances.append(pruned_performance)
                performance = get_fold_results(original_res, eval_criteria, False, fold, include_original_features,
                                               XofN_groupings,
                                               gen_XofN_time,
                                               training_time, path["dataset_name"])
                performances.append(performance)

    if len(performances) == 0:
        print("no XofN groupings were created in any fold")
    else:
        final_perf_df = pd.DataFrame(performances)
        averages = final_perf_df.groupby(["pruning", 'include_original_features', 'dataset'])[
            eval_criteria + ['nodes', 'leaves', 'groups',
                             'avg_group_features', 'gen_XofN_time', 'training_time']].mean().reset_index()
        print(averages)
        cv_results.append(averages)
        performances = []
# paths[0] - features[:10] desktop 0.29m
# laptop ??m
# desktop 11h


--- Running with label:'['se_C0009676', 'se_C0041657', 'se_C0002994', 'se_C0042571', 'se_C0004604', 'se_C0041834', 'se_C0085631', 'se_C0040822', 'se_C0042373', 'se_C0021053']' training_algorithm:'Jaccard' eval_criterion:'['averageAUROC', 'HammingLoss', 'SubsetAccuracy', 'RankingLoss', 'MacroPrecision', 'MacroRecall', 'MacroFOne']' max_size:'5' ---
Logs can be found in XofN_jaccard/logs/CPI+fingerprint_ten_mid_Jaccard_averageAUROC_HammingLoss_SubsetAccuracy_RankingLoss_MacroPrecision_MacroRecall_MacroFOne_5_logs.txt.

Fold 1/10 (CPI+fingerprint_ten_mid 1/3)

generate_XofN_list -> Generating groupings based on variance reduction:variance reduction.


🔄 Processing features: 100%|██████████| 2147/2147 [09:32<00:00,  3.75feat/s]


XofN groups with 2 features: 1
XofN groups with 5 features: 429
pruning: True, include_original_features: with_org, averageAUROC: 0.577625, HammingLoss: 0.34964, SubsetAccuracy: 0.1223, RankingLoss: 0.3364, MacroPrecision: 0.53147, MacroRecall: 0.3331, MacroFOne: 0.40451, 
pruning: False, include_original_features: with_org, averageAUROC: 0.5598991, HammingLoss: 0.42374, SubsetAccuracy: 0.035971, RankingLoss: 0.35431, MacroPrecision: 0.42807, MacroRecall: 0.48573, MacroFOne: 0.45186, 
pruning: True, include_original_features: no_org, averageAUROC: 0.5665876, HammingLoss: 0.35899, SubsetAccuracy: 0.10791, RankingLoss: 0.35769, MacroPrecision: 0.50846, MacroRecall: 0.30838, MacroFOne: 0.37328, 
pruning: False, include_original_features: no_org, averageAUROC: 0.5596082, HammingLoss: 0.42662, SubsetAccuracy: 0.057554, RankingLoss: 0.35056, MacroPrecision: 0.42153, MacroRecall: 0.47873, MacroFOne: 0.44634, 

Fold 2/10 (CPI+fingerprint_ten_mid 1/3)

generate_XofN_list -> Generating groupings

🔄 Processing features: 100%|██████████| 2147/2147 [09:29<00:00,  3.77feat/s] 


XofN groups with 2 features: 1
XofN groups with 5 features: 429
pruning: True, include_original_features: with_org, averageAUROC: 0.6028066, HammingLoss: 0.33885, SubsetAccuracy: 0.1223, RankingLoss: 0.35485, MacroPrecision: 0.51774, MacroRecall: 0.25558, MacroFOne: 0.34097, 
pruning: False, include_original_features: with_org, averageAUROC: 0.6090106, HammingLoss: 0.38633, SubsetAccuracy: 0.064748, RankingLoss: 0.35095, MacroPrecision: 0.44388, MacroRecall: 0.54877, MacroFOne: 0.49019, 
pruning: True, include_original_features: no_org, averageAUROC: 0.6162582, HammingLoss: 0.31511, SubsetAccuracy: 0.1295, RankingLoss: 0.33731, MacroPrecision: 0.59433, MacroRecall: 0.25205, MacroFOne: 0.35063, 
pruning: False, include_original_features: no_org, averageAUROC: 0.6330552, HammingLoss: 0.38201, SubsetAccuracy: 0.05036, RankingLoss: 0.31777, MacroPrecision: 0.45326, MacroRecall: 0.54691, MacroFOne: 0.49222, 

Fold 3/10 (CPI+fingerprint_ten_mid 1/3)

generate_XofN_list -> Generating grouping

...

XofN groups with 5 features: 108
pruning: True, include_original_features: with_org, averageAUROC: 0.5621502, HammingLoss: 0.38931, SubsetAccuracy: 0.10687, RankingLoss: 0.32664, MacroPrecision: 0.39202, MacroRecall: 0.33162, MacroFOne: 0.35833, 
pruning: False, include_original_features: with_org, averageAUROC: 0.6085708, HammingLoss: 0.41221, SubsetAccuracy: 0.061069, RankingLoss: 0.32583, MacroPrecision: 0.40961, MacroRecall: 0.56833, MacroFOne: 0.47499, 
pruning: True, include_original_features: no_org, averageAUROC: 0.6033387, HammingLoss: 0.33817, SubsetAccuracy: 0.1374, RankingLoss: 0.32621, MacroPrecision: 0.47554, MacroRecall: 0.34806, MacroFOne: 0.39784, 
pruning: False, include_original_features: no_org, averageAUROC: 0.5951183, HammingLoss: 0.42824, SubsetAccuracy: 0.030534, RankingLoss: 0.32829, MacroPrecision: 0.40149, MacroRecall: 0.60026, MacroFOne: 0.48003, 

Fold 9/10 (fingerprint_ten_mid 3/3)

generate_XofN_list -> Generating groupings based on variance reduction:var

🔄 Processing features: 100%|██████████| 540/540 [01:15<00:00,  7.12feat/s]


XofN groups with 5 features: 108
pruning: True, include_original_features: with_org, averageAUROC: 0.5788611, HammingLoss: 0.39466, SubsetAccuracy: 0.091603, RankingLoss: 0.34241, MacroPrecision: 0.42334, MacroRecall: 0.28564, MacroFOne: 0.33701, 
pruning: False, include_original_features: with_org, averageAUROC: 0.5853979, HammingLoss: 0.4374, SubsetAccuracy: 0.015267, RankingLoss: 0.33718, MacroPrecision: 0.41509, MacroRecall: 0.54588, MacroFOne: 0.46952, 
pruning: True, include_original_features: no_org, averageAUROC: 0.5750837, HammingLoss: 0.38397, SubsetAccuracy: 0.091603, RankingLoss: 0.36058, MacroPrecision: 0.45182, MacroRecall: 0.31201, MacroFOne: 0.3631, 
pruning: False, include_original_features: no_org, averageAUROC: 0.5807565, HammingLoss: 0.42443, SubsetAccuracy: 0.022901, RankingLoss: 0.36347, MacroPrecision: 0.42348, MacroRecall: 0.51858, MacroFOne: 0.46391, 

Fold 10/10 (fingerprint_ten_mid 3/3)

generate_XofN_list -> Generating groupings based on variance reduction:v

🔄 Processing features: 100%|██████████| 540/540 [01:15<00:00,  7.12feat/s]


XofN groups with 5 features: 108
pruning: True, include_original_features: with_org, averageAUROC: 0.5963773, HammingLoss: 0.36641, SubsetAccuracy: 0.10687, RankingLoss: 0.33095, MacroPrecision: 0.55909, MacroRecall: 0.29172, MacroFOne: 0.37624, 
pruning: False, include_original_features: with_org, averageAUROC: 0.6019363, HammingLoss: 0.41527, SubsetAccuracy: 0.038168, RankingLoss: 0.33245, MacroPrecision: 0.47795, MacroRecall: 0.56316, MacroFOne: 0.51632, 
pruning: True, include_original_features: no_org, averageAUROC: 0.5756279, HammingLoss: 0.39008, SubsetAccuracy: 0.14504, RankingLoss: 0.35575, MacroPrecision: 0.509, MacroRecall: 0.23362, MacroFOne: 0.31623, 
pruning: False, include_original_features: no_org, averageAUROC: 0.609594, HammingLoss: 0.39771, SubsetAccuracy: 0.053435, RankingLoss: 0.32899, MacroPrecision: 0.49634, MacroRecall: 0.51102, MacroFOne: 0.50111, 
   pruning include_original_features              dataset  averageAUROC  \
0    False                    no_org  f

In [5]:
# All results
save_path = "XofN_jaccard/"
final_grouped_res = pd.DataFrame()
for df in cv_results:
    final_grouped_res = pd.concat([final_grouped_res, df])
final_grouped_res.sort_values(by='dataset', ascending=False, inplace=True)
final_grouped_res = final_grouped_res.reset_index(drop=True)
final_grouped_res.to_csv(save_path + "all_results.csv")
final_grouped_res

,pruning,include_original_features,dataset,averageAUROC,HammingLoss,SubsetAccuracy,RankingLoss,MacroPrecision,MacroRecall,MacroFOne,nodes,leaves,groups,avg_group_features,gen_XofN_time,training_time
0,False,no_org,fingerprint_ten_mid,0.593201,0.405204,0.041939,0.336606,0.434109,0.521986,0.470605,989.8,495.4,108.0,5.000000,75.726180,0.670816
1,False,with_org,fingerprint_ten_mid,0.577148,0.422882,0.039671,0.342832,0.415210,0.511857,0.455048,994.2,497.6,108.0,5.000000,75.726180,1.253825
2,True,no_org,fingerprint_ten_mid,0.584619,0.353771,0.117481,0.342575,0.483136,0.259385,0.331308,98.4,49.7,108.0,5.000000,75.726180,0.670816
3,True,with_org,fingerprint_ten_mid,0.574991,0.368042,0.102995,0.342239,0.454493,0.266533,0.329183,109.6,55.3,108.0,5.000000,75.726180,1.253825
4,False,no_org,CPI_ten_mid,0.580407,0.413957,0.045168,0.354359,0.443703,0.505486,0.469000,772.4,386.7,322.0,4.990683,349.730067,1.086974
5,False,with_org,CPI_ten_mid,0.589332,0.409292,0.052400,0.346213,0.446657,0.508703,0.472155,793.4,397.2,322.0,4.990683,349.730067,3.065245
6,True,no_org,CPI_ten_mid,0.580660,0.352377,0.115651,0.371796,0.515036,0.271272,0.319582,71.0,36.0,322.0,4.990683,349.730067,1.086974
7,True,with_org,CPI_ten_mid,0.593023,0.357897,0.108435,0.367881,0.501032,0.319519,0.383326,88.0,44.5,322.0,4.990683,349.730067,3.065245
8,False,no_org,CPI+fingerprint_ten_mid,0.579709,0.410430,0.046142,0.351905,0.420784,0.487624,0.449191,1029.2,515.1,430.0,4.993023,574.383363,1.181200
9,False,with_org,CPI+fingerprint_ten_mid,0.586958,0.404702,0.048316,0.346406,0.428724,0.504924,0.461434,1025.2,513.1,430.0,4.993023,574.383363,3.951507


In [6]:
# Table ready (with pruning, rounded, compact)
save_path = "XofN_jaccard/"
rounded_final_grouped_res = pd.read_csv(save_path + "all_results.csv", index_col=0)
res = pd.read_csv(save_path + "all_results.csv", index_col=0)
table_results = get_table_results(res, eval_criteria, get_dataset_paths())
table_results.to_csv(save_path + "table_results.csv")
table_results

,include_original_features,dataset,averageAUROC,HammingLoss,SubsetAccuracy,RankingLoss,MacroPrecision,MacroRecall,MacroFOne,Nodes; Leaves,#XofN; #Feat/XofN,# Ung. Feats,XofN time (s); PCT tr. time (s)
2,no_org,fingerprint_ten_mid,0.585,0.354,0.117,0.343,0.483,0.259,0.331,98.4; 49.7,108.0; 5.0,0.0,75.7; 0.7
3,with_org,fingerprint_ten_mid,0.575,0.368,0.103,0.342,0.454,0.267,0.329,109.6; 55.3,108.0; 5.0,0.0,75.7; 1.3
6,no_org,CPI_ten_mid,0.581,0.352,0.116,0.372,0.515,0.271,0.320,71.0; 36.0,322.0; 5.0,0.0,349.7; 1.1
7,with_org,CPI_ten_mid,0.593,0.358,0.108,0.368,0.501,0.320,0.383,88.0; 44.5,322.0; 5.0,0.0,349.7; 3.1
10,no_org,CPI+fingerprint_ten_mid,0.588,0.346,0.127,0.354,0.505,0.263,0.340,91.6; 46.3,430.0; 5.0,0.0,574.4; 1.2
11,with_org,CPI+fingerprint_ten_mid,0.590,0.347,0.128,0.348,0.501,0.272,0.347,109.0; 55.0,430.0; 5.0,0.0,574.4; 4.0
